## Open In Colab\n\n[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/risk-uq-suite/notebooks/risk_model_training_colab.ipynb)


# Risk/UQ Paper Track: Risk Model Training

## Objective
Train and calibrate the CRAR risk ensemble with resumable execution and persistent artifacts.


## Hypotheses Being Tested
- **H1:** Post-hoc temperature scaling improves reliability (ECE/NLL) vs raw probabilities.
- **H2:** Calibrated `failure_proxy_h15` remains usable under configured shift suites.
- **H3:** Resume protocol prevents progress loss under Colab runtime interrupts.


## Step 1 - Deterministic Bootstrap Constants
Define runtime/setup constants (idempotent).


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
EXPERIMENT_SLUG = 'risk-uq-suite'
EXPERIMENT_CONFIG_PATH = f'{REPO_DIR}/configs/experiments/{EXPERIMENT_SLUG}.json'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))


## Step 2 - Storage Setup
Mount Drive and verify writable persistent root.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('[storage] google.colab drive mount unavailable:', exc)

drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    raise RuntimeError('Drive root not found at /content/drive/MyDrive. Ensure Google Drive is mounted.')

required_root = Path(REQUIRED_DRIVE_FOLDER)
required_root.mkdir(parents=True, exist_ok=True)
probe = required_root / '.risk_uq_storage_probe'
probe.write_text('ok')
probe.unlink(missing_ok=True)
print('[storage] ready:', required_root)


## Step 3 - Repo Sync + Runtime Bootstrap
Clone/sync repo and initialize deterministic runtime bootstrap.


In [ ]:
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


## Step 4 - Configuration + Run Context
Load `configs/experiments/risk-uq-suite.json` and resolve run context.


In [ ]:
from src.workflows import initialize_run_context, load_experiment_config, report_run_context

EXPERIMENT_CFG = load_experiment_config(
    slug=EXPERIMENT_SLUG,
    repo_root=REPO_DIR,
    default_on_missing=False,
)
run_cfg = dict(EXPERIMENT_CFG.get('run', {}))

# Mandatory contract fields
RUN_NAME = str(run_cfg.get('run_name', '')).strip()
RUN_PREFIX = str(run_cfg.get('run_prefix', 'risk_uq')).strip() or 'risk_uq'
PERSIST_ROOT = str(run_cfg.get('persist_root', '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1')).strip()
N_SHARDS = int(max(1, int(run_cfg.get('n_shards', 1))))
SHARD_ID = run_cfg.get('shard_id', 'auto')
RESUME_FROM_EXISTING = bool(run_cfg.get('resume_from_existing', True))
RUN_ENABLED = bool(run_cfg.get('run_enabled', True))

RUN_TAG_PREFIX = str(run_cfg.get('run_tag_prefix', RUN_PREFIX)).strip() or RUN_PREFIX
RUN_MODE_CFG = str(run_cfg.get('run_mode', 'auto')).strip().lower() or 'auto'
RUN_MODE = RUN_MODE_CFG if RESUME_FROM_EXISTING else 'fresh'
RUN_TAG = RUN_NAME

AUTO_RUN_MAIN_LOOP_WHEN_READY = False
RUN_MAIN_LOOP_OVERRIDE = False
WARN_ON_CONFIG_DRIFT = True

run_context = initialize_run_context(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    auto_run_main_loop_when_ready=bool(AUTO_RUN_MAIN_LOOP_WHEN_READY),
    run_main_loop_override=RUN_MAIN_LOOP_OVERRIDE,
    run_tag_prefix=RUN_TAG_PREFIX,
    planner_backend='latentdriver',
    planner_surprise_name='predictive_seq_w2',
    auto_generate_run_tag_if_empty=True,
    resume_mode=RUN_MODE,
    warn_on_config_drift=bool(WARN_ON_CONFIG_DRIFT),
)

cfg = run_context.cfg
search_cfg = run_context.search_cfg
RUN_TAG = str(run_context.run_tag)
RUN_NAME = str(RUN_NAME or RUN_TAG)
SHARD_ID = int(run_context.shard_id)
run_prefix = run_context.run_prefix

print('EXPERIMENT_CONFIG_PATH =', EXPERIMENT_CONFIG_PATH)
print('run_prefix (flat artifacts) =', run_prefix)
print('RUN_PREFIX/RUN_NAME (contract dir) =', RUN_PREFIX, RUN_NAME)
print('RUN_ENABLED =', RUN_ENABLED, ' RESUME_FROM_EXISTING =', RESUME_FROM_EXISTING)
report_run_context(run_context, display_fn=display)


## Step 5 - Risk/UQ Training Knobs
Keep experiment controls centralized in one cell.


In [ ]:
cfg.risk_dataset_build = True
cfg.risk_dataset_candidate_count = 8
cfg.risk_dataset_control_horizon_steps = 6
cfg.risk_dataset_label_horizons = (5, 10, 15)
cfg.risk_dataset_events = ('collision', 'offroad', 'failure_proxy')

cfg.risk_model_ensemble_size = 5
cfg.risk_model_hidden_dims = (128, 128)
cfg.risk_model_dropout = 0.10
cfg.risk_model_learning_rate = 1e-3
cfg.risk_model_batch_size = 1024
cfg.risk_model_max_epochs = 50
cfg.risk_model_patience = 8
cfg.risk_model_checkpoint_every_epochs = 1
cfg.risk_model_resume_from_checkpoints = True

cfg.risk_calibration_method = 'temperature'
cfg.risk_conformal_alpha = 0.10
cfg.risk_control_fail_budget = 0.20

cfg.uq_shift_suites = ('nominal_clean', 'hist_prim_shift', 'fut_prim_shift', 'hist_rmv_shift', 'high_interaction_holdout')
cfg.uq_eval_probability_bins = 15
cfg


## Step 6 - Fast-Fail Validation
Run objective-aligned gates: risk smoke probe + preflight checks.


In [ ]:
import pandas as pd
import numpy as np
from src.workflows import (
    build_full_simulation_context,
    has_existing_risk_model_artifacts,
    load_existing_risk_dataset_artifact,
    load_notebook_contract_manifest,
    run_risk_training_notebook_gates,
)

existing_dataset_df = load_existing_risk_dataset_artifact(cfg.run_prefix)
existing_model_ready = has_existing_risk_model_artifacts(cfg.run_prefix)
if RUN_MODE == 'fresh':
    existing_dataset_df = pd.DataFrame()

needs_simulation_context = bool(existing_dataset_df.empty)
runner = None
eval_idx = None
reference_df = None

print('existing_dataset_rows =', len(existing_dataset_df))
print('existing_model_ready =', existing_model_ready)
print('needs_simulation_context =', needs_simulation_context)

if needs_simulation_context:
    sim_bundle = build_full_simulation_context(cfg=cfg, n_shards=N_SHARDS, shard_id=SHARD_ID)
    runner = sim_bundle.runner
    eval_idx = sim_bundle.eval_idx
    reference_df = sim_bundle.reference_df

    gate_bundle = run_risk_training_notebook_gates(
        runner=runner,
        cfg=cfg,
        eval_idx=eval_idx,
        probe_shift_suite='nominal_clean',
    )
    risk_probe_pass = bool(gate_bundle.get('risk_probe_pass', False))
    preflight_pass = bool(gate_bundle.get('preflight_pass', False))
    overall_pass = bool(gate_bundle.get('overall_pass', False))

    display(gate_bundle.get('risk_probe_summary_df', pd.DataFrame()))
    display(gate_bundle.get('preflight_df', pd.DataFrame()))
else:
    gate_bundle = {'failure_reasons': []}
    numeric_cols = [c for c in existing_dataset_df.columns if str(existing_dataset_df[c].dtype) != 'object']
    finite_ok = False
    if len(existing_dataset_df) > 0 and len(numeric_cols) > 0:
        sample = existing_dataset_df.loc[:, numeric_cols].head(4096).to_numpy(dtype=float)
        finite_ok = bool(np.isfinite(sample).all())

    prior_manifest = load_notebook_contract_manifest(cfg.run_prefix)
    prior_preflight = bool(prior_manifest.get('preflight_pass', False))

    risk_probe_pass = bool((len(existing_dataset_df) > 0) and finite_ok)
    preflight_pass = bool(prior_preflight or existing_model_ready)
    overall_pass = bool(risk_probe_pass and preflight_pass)

print('risk_probe_pass =', risk_probe_pass)
print('preflight_pass =', preflight_pass)
print('overall_pass =', overall_pass)

if not overall_pass:
    raise RuntimeError(f"Risk notebook gates failed: {gate_bundle.get('failure_reasons', [])}")


## Step 7 - Manifest + Contract Layout (Gates Passed)
Write notebook stage manifest and mirror contract folder layout.


In [ ]:
from src.workflows import write_contract_storage_mirror, write_notebook_contract_manifest

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='risk_model_training_colab',
    stage='gates_passed',
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    quick_probe_pass=bool(risk_probe_pass),
    preflight_pass=bool(preflight_pass),
)

contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage='gates_passed',
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
)
print('manifest_path =', manifest_path)
print('contract_run_dir =', contract_paths['contract_run_dir'])


## Step 8 - Main Execution (Resume-Aware Training)
Training runs only when `RUN_ENABLED=True`.


In [ ]:
from src.workflows import (
    has_existing_risk_model_artifacts,
    has_existing_risk_training_checkpoints,
    run_risk_training_flow,
)

bundle = None
if not bool(RUN_ENABLED):
    print('[main] skipped: RUN_ENABLED=False')
else:
    existing_model_ready = has_existing_risk_model_artifacts(cfg.run_prefix)
    existing_checkpoint_ready = has_existing_risk_training_checkpoints(cfg.run_prefix)

    print('existing_model_ready =', existing_model_ready)
    print('existing_checkpoint_ready =', existing_checkpoint_ready)

    if existing_dataset_df.empty:
        print('[risk-train] no persisted dataset found; building from current runner context...')
        bundle = run_risk_training_flow(
            cfg=cfg,
            runner=runner,
            run_prefix=cfg.run_prefix,
            resume_mode=RUN_MODE,
        )
    else:
        print(f'[risk-train] using persisted dataset rows={len(existing_dataset_df)}')
        bundle = run_risk_training_flow(
            cfg=cfg,
            dataset_df=existing_dataset_df,
            run_prefix=cfg.run_prefix,
            resume_mode=RUN_MODE,
        )

if bundle is not None:
    print('loaded_from_existing =', bundle.loaded_from_existing)
    print('dataset_rows =', len(bundle.dataset_bundle.dataset_df))
    print('artifact_keys =', sorted(bundle.artifact_paths.keys()))


## Step 9 - Evaluation/Diagnostics


In [ ]:
if bundle is None:
    print('[report] no run output because RUN_ENABLED=False')
else:
    display(bundle.training_bundle.train_summary.head(20))
    if bundle.calibration_bundle is not None:
        display(bundle.calibration_bundle.summary_df)
        display(bundle.calibration_bundle.reliability_df.head(20))


## Step 10 - Export + Manifest (Completed)
Update notebook manifest and contract folder outputs.


In [ ]:
if bundle is None:
    stage_name = 'risk_training_skipped'
    artifact_paths = {}
    metrics_path = None
    extra = {'run_skipped': 1}
else:
    stage_name = 'risk_training_completed'
    artifact_paths = dict(bundle.artifact_paths)
    metrics_path = str(bundle.artifact_paths.get('risk_train_summary', '')) or None
    extra = {
        'loaded_from_existing': int(bool(bundle.loaded_from_existing)),
        'artifact_keys': sorted(list(bundle.artifact_paths.keys())),
    }

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='risk_model_training_colab',
    stage=stage_name,
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    quick_probe_pass=bool(risk_probe_pass),
    preflight_pass=bool(preflight_pass),
    extra_fields=extra,
)

contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage=stage_name,
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
    artifact_paths=artifact_paths,
    metrics_csv_path=metrics_path,
    extra_fields=extra,
)

print('manifest_path =', manifest_path)
print('contract_run_manifest =', contract_paths['contract_run_manifest'])
